# `TableParser`, `DelimTxtParser`, and `NDAParser`

A `TableParser` reads tabular data from a file and normalises the column names to BDF standards.
Two concrete parsers are shown here:
- **`DelimTxtParser`** – reads delimiter-separated text files (CSV, TSV, etc.)
- **`NDAParser`** – reads Neware's binary `.nda` / `.ndax` format

Both expose the same interface: `matches_ext()`, `read_column_headings()`, `normalizer_score()`, and `read()`.

In [ ]:
from pathlib import Path
from bdf.table_parsers import DelimTxtParser, NDAParser
from bdf.table_normalizers import TableNormalizer, Syn, DateTimeSyn
from bdf.file_utils import resolve_source

BIOLOGIC_URL = (
    "https://zenodo.org/api/records/18986774/files/"
    "SINTEF__NaCR32140-MP10-04__2025-08-25__GITT_0p05C_25degC__BioLogic.mpt/content"
)
NEWARE_NDA_URL = (
    "https://zenodo.org/api/records/18986774/files/"
    "SINTEF__G20M7-202512-Gru6mV__20251228__C30__25degC__Neware.nda/content"
)

## `DelimTxtParser` — delimited text files

Construct a `DelimTxtParser` with a `TableNormalizer` that maps raw column names to BDF fields.
The Biologic BT-Lab file has tab-separated columns including `time/s`, `Ecell/V`, `I/mA`, and `cycle number`.
Unit-template syns like `Syn(hdr='Ecell/{unit}')` match any column whose name fits the pattern with a recognised unit.
Matching is case-sensitive, so the syn root must match the source header's casing exactly outside the `{unit}` slot.

In [ ]:
biologic_file = resolve_source(BIOLOGIC_URL)

biologic_parser = DelimTxtParser(
    normalizer=TableNormalizer(
        test_time_second=(Syn(hdr="time/{unit}"),),
        voltage_volt=(Syn(hdr="Ecell/{unit}"),),
        current_ampere=(Syn(hdr="I/{unit}"),),
        cycle_count=(Syn(hdr="cycle number"),),
    )
)
biologic_parser

`matches_ext()` checks the file extension against the parser's accepted types

In [ ]:
print("matches .txt: ", biologic_parser.matches_ext(".txt"))
print("matches .xlsx:", biologic_parser.matches_ext(".xlsx"))

`read_column_headings()` sniffs only the header row — no data rows loaded


In [ ]:
biologic_parser.read_column_headings(biologic_file)

`normalizer_score()` counts how many columns the normalizer can resolve


In [ ]:
biologic_parser.normalizer_score(biologic_file)

`read()` returns a normalised polars LazyFrame with BDF-standard column names

In [ ]:
biologic_parser.read(biologic_file).collect()

## Graceful degradation — empty normalizer preserves source column names

When a `TableNormalizer()` with no synonyms is supplied, `read()` still works but
keeps the original column names from the file. This is useful for exploring an unfamiliar file.

In [ ]:
raw_parser = DelimTxtParser(normalizer=TableNormalizer())
raw_parser.read(biologic_file, validate=False).collect()

## `NDAParser` — Neware binary files

`NDAParser` reads Neware's binary `.nda` / `.ndax` format via `fastnda`.
Its extension handling targets the binary format, the opposite of `DelimTxtParser`: `.nda` returns `True`, `.csv` returns `False`.

In [ ]:
neware_file = resolve_source(NEWARE_NDA_URL)

neware_parser = NDAParser(
    normalizer=TableNormalizer(
        test_time_second=(Syn(hdr="total_time_{unit}"),),
        voltage_volt=(Syn(hdr="voltage_{unit}"),),
        current_ampere=(Syn(hdr="current_{unit}"),),
        cycle_count=(Syn(hdr="cycle_count"),),
    ),
)
neware_parser

In [ ]:
print("matches .nda: ", neware_parser.matches_ext(".nda"))
print("matches .csv: ", neware_parser.matches_ext(".csv"))

In [ ]:
neware_parser.read_column_headings(neware_file)

In [ ]:
neware_parser.read(neware_file)